[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/benchuangxd/CSC3109-T16-Project/blob/main/notebooks/03_resnet18.ipynb)

# ResNet-18

**CSC3109 - Machine Learning | Team 16 — Member 2**

In [1]:
# ── Google Colab Setup ──────────────────────────────────────────────────────
# Run this cell first when using Google Colab. No effect when running locally.
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import subprocess
    from pathlib import Path

    REPO_URL  = "https://github.com/benchuangxd/CSC3109-T16-Project.git"
    REPO_PATH = Path("/content/CSC3109-T16-Project")

    if not REPO_PATH.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_PATH)], check=True)
    else:
        print(f"Repo already exists at {REPO_PATH}")

    %cd /content/CSC3109-T16-Project
    %pip install -q -r requirements.txt
    print("Colab setup complete.")
else:
    print("Running locally — skipping Colab setup.")

/content/CSC3109-T16-Project
Colab setup complete.


In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.config import DATA_DIR, CLASS_NAMES, NUM_CLASSES, IMAGE_SIZE, BATCH_SIZE, NUM_EPOCHS, SEED, TRAIN_RATIO
from src.utils import set_seed, get_device

set_seed(SEED)
DEVICE = get_device()
print(f"ROOT   : {ROOT}")
print(f"Device : {DEVICE}")
print(f"Classes: {CLASS_NAMES}")
print(f"Split  : {TRAIN_RATIO:.0%} train / {1-TRAIN_RATIO:.0%} val")
print("Policy : 70/30 split inside each class folder in data/set 16")

ROOT   : /content/CSC3109-T16-Project
Device : cpu
Classes: ['beach', 'ferry_terminal', 'harbor', 'river']
Split  : 70% train / 30% val
Policy : 70/30 split inside each class folder in data/set 16


## 1. Data Loading

The dataloader applies the shared project split: a 70/30 split inside each class folder in `data/set 16`. This keeps every class at 490 train and 210 validation images.


In [3]:
from src.dataset import get_dataloaders

train_loader, val_loader, classes = get_dataloaders(
    root        = ROOT,
    data_dir    = DATA_DIR,
    batch_size  = BATCH_SIZE,
    image_size  = IMAGE_SIZE,
    train_ratio = TRAIN_RATIO,
    seed        = SEED,
    num_workers = 2,
)

def subset_class_counts(subset, class_names):
    targets = subset.dataset.targets
    counts = {cls: 0 for cls in class_names}
    for idx in subset.indices:
        counts[class_names[targets[idx]]] += 1
    return counts

print(f"Classes       : {classes}")
print(f"Split policy  : {TRAIN_RATIO:.0%}/{1-TRAIN_RATIO:.0%} inside each class folder")
print(f"Train batches : {len(train_loader)}  ({len(train_loader.dataset)} images)")
print(f"Val batches   : {len(val_loader)}  ({len(val_loader.dataset)} images)")
print(f"Train counts  : {subset_class_counts(train_loader.dataset, classes)}")
print(f"Val counts    : {subset_class_counts(val_loader.dataset, classes)}")


Classes       : ['beach', 'ferry_terminal', 'harbor', 'river']
Split policy  : 70%/30% inside each class folder
Train batches : 62  (1960 images)
Val batches   : 27  (840 images)
Train counts  : {'beach': 490, 'ferry_terminal': 490, 'harbor': 490, 'river': 490}
Val counts    : {'beach': 210, 'ferry_terminal': 210, 'harbor': 210, 'river': 210}


## 2. Model Setup

ResNet-18 stacks residual blocks with skip connections that let gradients flow through 18 layers without vanishing. ImageNet pretrained model is used, weights and replace only the final fully-connected layer to output 4 classes


In [4]:
from src.models import get_resnet18
from src.utils import count_parameters

model = get_resnet18(num_classes=NUM_CLASSES, pretrained=True)
model = model.to(DEVICE)

total_params = count_parameters(model)
print(f"Trainable parameters : {total_params / 1e6:.2f} M")
print(f"Model head           : {model.fc}")
print(f"Running on           : {DEVICE}")


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 130MB/s]


Trainable parameters : 11.18 M
Model head           : Linear(in_features=512, out_features=4, bias=True)
Running on           : cpu
